In [ ]:
import torch
import numpy as np
from datasets import load_dataset, concatenate_datasets
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from trl import SFTTrainer
import json

# Step 1: Check environment
print(f"NumPy version: {np.__version__}")  # Should be <2.0.0
print(f"Torch device: {'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'}")
import transformers
print(f"transformers version: {transformers.__version__}")
import trl
print(f"trl version: {trl.__version__}")

# Step 2: Load Tokenizer and Model
model_id = "TinyLlama/TinyLlama_v1.1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Define chat template
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{% if message['role'] == 'user' %}"
    "User: {{ message['content'] }} <|end|>\n"
    "{% elif message['role'] == 'assistant' %}"
    "Assistant: {{ message['content'] }} <|end|>\n"
    "{% endif %}"
    "{% endfor %}"
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True
)

# Step 3: Preprocess Dataset
def format_dataset(example):
    if "messages" in example:
        messages = example["messages"]
    elif "Context" in example and "Response" in example:
        messages = [
            {"role": "user", "content": example["Context"]},
            {"role": "assistant", "content": example["Response"]}
        ]
    else:
        raise ValueError(f"Unknown dataset format: {example}")
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    # Truncate if needed (alternative to max_seq_length)
    tokenized = tokenizer(text, truncation=True, max_length=512, add_special_tokens=False)
    return {"text": tokenized["input_ids"]}

# Load and preprocess datasets
dataset = load_dataset("json", data_files="dataset_train_line.jsonl")
dataset = dataset["train"].train_test_split(test_size=0.1)
train_dataset = dataset["train"].map(format_dataset)
eval_dataset = dataset["test"].map(format_dataset)

# Add medical dataset
medical_dataset = load_dataset("Amod/mental_health_counseling_conversations")["train"].map(format_dataset, remove_columns=["Context", "Response"])
train_dataset = concatenate_datasets([train_dataset, medical_dataset])

# Step 4: Define Training Arguments
training_args = TrainingArguments(
    output_dir="./cbt_cot_model_new",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    save_steps=200,
    logging_steps=20,
    learning_rate=2e-5,
    fp16=False,
    bf16=True if torch.backends.mps.is_available() else False,
    evaluation_strategy="steps",
    eval_steps=200,
    save_total_limit=2,
    #max_seq_length=512,
)

# Step 5: Fine-Tune
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=training_args,
    #max_seq_length=512,
)

trainer.train()
trainer.save_model()

# Step 6: Inference on test.jsonl
try:
    test_dataset = load_dataset("json", data_files="test.jsonl")["train"]
    test_dataset = test_dataset.map(format_dataset)
except FileNotFoundError:
    print("test.jsonl not found. Skipping inference.")
    test_dataset = None

if test_dataset:
    pipe = pipeline("text-generation", model="./cbt_cot_model_new", tokenizer=tokenizer)
    responses = []

    for example in test_dataset:
        prompt = example["text"].split("Assistant:")[0]
        try:
            response = pipe(
                prompt,
                max_new_tokens=400,
                do_sample=True,
                temperature=0.7
            )[0]["generated_text"]
            responses.append({"prompt": prompt, "response": response})
        except Exception as e:
            responses.append({"prompt": prompt, "response": f"Error: {e}"})

    # Save responses
    with open("test_responses.jsonl", "w") as f:
        for r in responses:
            json.dump(r, f)
            f.write("\n")

    # Print sample
    print("Sample response:")
    print(responses[0] if responses else "No responses generated.")
else:
    print("No test dataset available. Training completed.")